# Day 4: Data loaders, augmentation check, overfit test
**Use a GPU runtime** (Runtime > Change runtime type > T4 GPU) before running.

Three jobs: (1) build the loaders with augmentation, (2) *look at* what the model will see, (3) prove the pipeline can learn at all by memorising 32 images. If step 3 fails, something upstream is broken and a long training run would waste hours.

It expects the Day 3 folders in `/content/maize_split`. If the runtime was reset, it restores them from the Drive zip made in Day 3's optional step (`maize_split.zip` in the top of My Drive).

## 1. Setup and find the data

In [ ]:
import os, time, random, zipfile
import numpy as np
import matplotlib.pyplot as plt
import torch, torchvision
from torch import nn
from torchvision import datasets, transforms
from PIL import Image

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "NO GPU: switch the runtime to a T4 GPU")

ROOT = "/content/maize_split"
if not os.path.isdir(ROOT + "/train"):
    from google.colab import drive
    drive.mount("/content/drive")
    z = "/content/drive/MyDrive/maize_split.zip"
    assert os.path.exists(z), "No /content/maize_split and no maize_split.zip on Drive: re-run the Day 3 notebook (and its last, optional cell)."
    os.makedirs(ROOT, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(ROOT)
print("data folders:", sorted(os.listdir(ROOT)))


## 2. Transforms and loaders
- **Train:** random crop (keeps 60 to 100% of the image, so a lesion is rarely cropped out), flips, rotation up to 20 degrees, and *mild* colour jitter. Colour is a real symptom here (yellowing, streaks), so brightness and contrast vary a little, saturation a little, hue barely at all.
- **Val/test:** no randomness: resize to 256 on the short side, centre crop 224.
- Everything is normalised with the ImageNet mean and std, which the pretrained ResNet expects.

The rotation leaves small black corners; that is fine.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(ROOT + "/train", train_tf)
val_ds = datasets.ImageFolder(ROOT + "/val", eval_tf)
test_ds = datasets.ImageFolder(ROOT + "/test", eval_tf)
assert train_ds.classes == val_ds.classes == test_ds.classes, "class folders differ between splits"
classes = train_ds.classes
print("classes:", train_ds.class_to_idx)
print("images: train", len(train_ds), "| val", len(val_ds), "| test", len(test_ds), "(Day 3 summary said 7485 / 850 / 845)")
for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    print(f"  {name}:", {classes[k]: int(v) for k, v in zip(*np.unique(ds.targets, return_counts=True))})

BATCH = 32
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=(device == "cuda"))
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)


## 3. Look at what the model sees
Check every one of these before moving on:
- The labels above each image match what you can see (healthy leaves look healthy; MLN vs MSV is hard even for people, so don't expect to tell them apart, but a clearly healthy leaf labelled as diseased, or the reverse, is a red flag).
- The crops still show the leaf and, where visible, the lesions.
- Colours look natural, not neon or washed out.
- Images aren't squashed or stretched.
- The same image augmented eight times (second picture) looks like eight plausible photos of one leaf.

In [ ]:
mean_t = torch.tensor(MEAN).view(1, 3, 1, 1); std_t = torch.tensor(STD).view(1, 3, 1, 1)

def show(x, y, n=16, cols=8, title=""):
    x = (x[:n].cpu() * std_t + mean_t).clamp(0, 1)
    rows = int(np.ceil(len(x) / cols))
    fig, ax = plt.subplots(rows, cols, figsize=(2 * cols, 2.3 * rows), squeeze=False)
    for a in ax.ravel():
        a.axis("off")
    for k in range(len(x)):
        ax.ravel()[k].imshow(x[k].permute(1, 2, 0).numpy())
        ax.ravel()[k].set_title(classes[int(y[k])], fontsize=8)
    plt.suptitle(title); plt.tight_layout(); plt.show()

x, y = next(iter(train_loader))
print("batch:", tuple(x.shape), x.dtype, "| labels in batch:", torch.bincount(y, minlength=len(classes)).tolist())
print("per-channel mean (should be near 0):", x.mean((0, 2, 3)).numpy().round(2), "| std (near 1):", x.std((0, 2, 3)).numpy().round(2))
show(x, y, 16, title="Augmented training batch")

idx = random.randrange(len(train_ds))
path, lab = train_ds.samples[idx]
img = Image.open(path).convert("RGB")
xs = torch.stack([train_tf(img) for _ in range(8)])
show(xs, [lab] * 8, 8, title=f"One image ({os.path.basename(path)}), augmented 8 times")

xv, yv = next(iter(val_loader))
show(xv, yv, 16, title="Validation batch (no augmentation)")


## 4. How fast does data load?
If loading is slower than the GPU, training time is set by the CPU (Colab gives you 2 cores). This gives you a rough epoch time for planning Day 5.

In [ ]:
t0 = time.time(); n = 0
for i, (xb, yb) in enumerate(train_loader):
    n += len(xb)
    if i == 19:
        break
rate = n / (time.time() - t0)
print(f"loading: about {rate:.0f} images/s -> roughly {len(train_ds) / rate / 60:.1f} min per epoch if loading is the bottleneck")


## 5. Overfit test: can the pipeline learn anything?
Take 32 training images, no augmentation, and train a pretrained ResNet18 (new 3-class head) on just those until it memorises them.

**Pass:** loss starts near 1.1 (that's ln 3, a model guessing among 3 classes) and falls below about 0.05, with 100% accuracy on those 32 images.
**Fail:** the loss stays high, or jumps around, or accuracy is stuck. Likely causes: labels not matching images, a learning rate that's too high, the model left in the wrong mode, or the data and labels being shuffled apart. Paste me the printout and we'll find it.

In [ ]:
overfit_ds = datasets.ImageFolder(ROOT + "/train", eval_tf)             # same files, no augmentation
sel = random.Random(SEED).sample(range(len(overfit_ds)), 32)
xb, yb = next(iter(torch.utils.data.DataLoader(torch.utils.data.Subset(overfit_ds, sel), batch_size=32)))
xb, yb = xb.to(device), yb.to(device)
print("labels in the 32:", torch.bincount(yb.cpu(), minlength=len(classes)).tolist())

model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(classes))
model = model.to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

losses = []
for step in range(80):
    model.train()
    opt.zero_grad()
    out = model(xb)
    loss = loss_fn(out, yb)
    loss.backward(); opt.step()
    losses.append(loss.item())
    if step % 10 == 0 or step == 79:
        print(f"step {step:2d}  loss {loss.item():.4f}  acc {(out.argmax(1) == yb).float().mean().item():.2f}")

model.eval()
with torch.no_grad():
    final_acc = (model(xb).argmax(1) == yb).float().mean().item()
print(f"\nfinal (eval mode): loss {losses[-1]:.4f}, accuracy {final_acc:.2f}, first-step loss {losses[0]:.3f} (ln 3 = 1.099)")
passed = losses[-1] < 0.05 and final_acc == 1.0
print("OVERFIT TEST:", "PASSED" if passed else "NOT PASSED: see the notes above this cell")
plt.plot(losses); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss"); plt.title("Overfit test on 32 images"); plt.show()


## 6. Save a summary
Paste `day4_summary.md` back into the chat.

In [ ]:
lines = [
    "# Day 4 summary",
    f"- device: {device}",
    f"- classes: {train_ds.class_to_idx}",
    f"- images: train {len(train_ds)}, val {len(val_ds)}, test {len(test_ds)}",
    f"- augmentation: RandomResizedCrop(224, scale 0.6-1.0), h/v flip, rotation 20, ColorJitter(0.2, 0.2, 0.2, 0.02); eval: resize 256 + centre crop 224",
    f"- loading speed: about {rate:.0f} images/s (~{len(train_ds) / rate / 60:.1f} min/epoch if loading-bound)",
    f"- overfit test: first loss {losses[0]:.3f}, last loss {losses[-1]:.4f}, accuracy {final_acc:.2f} -> {'PASSED' if passed else 'NOT PASSED'}",
    "- notes from the batch pictures (fill in): labels plausible? crops keep the leaf? colours natural?",
]
open("/content/day4_summary.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
try:
    from google.colab import files
    files.download("/content/day4_summary.md")
except ImportError:
    pass
